## tl;dr

The Gemini batch contains authentic, reviewable Google US evidence for all nine queries: 9 HTML files, 9 full-page PNG screenshots, 9 JSON query keys, and 4 reports. The parser captured 80 visible first-page organic result rows, not 90 Top-10 rows; six queries contain only 7-9 visible organic rows because Google interleaves AI Overview, video, forum, and ad modules. Reports are usable only after interpretation repair: QR event intent and Lead AI intent are mixed, and Typeform SERPs are more first-party-product-heavy than the reported 40/40/20 split.

## Context & Methods

Decision: whether QR, Lead, and Typeform research may exit the Research lane. The notebook checks file completeness, JSON structure, unique/canonical URLs, missing snippets, US query markers, and PNG dimensions. Intent interpretation remains a manual screenshot review and is recorded in the Codex evidence review.

In [1]:
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
import json
import struct
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != 'AIFactory':
    ROOT = next((p for p in [ROOT, *ROOT.parents] if (p / 'ProjectDocs').exists()), ROOT)
BATCH = ROOT / 'ProjectDocs/Operations/seo_growth_production_loop/research_batch_2026-07-03'
EVIDENCE = BATCH / 'evidence'
SERP = json.loads((BATCH / 'serp_data.json').read_text())
print({'batch': str(BATCH), 'queries': len(SERP)})

{'batch': '/Users/mike/Documents/AIFactory/ProjectDocs/Operations/seo_growth_production_loop/research_batch_2026-07-03', 'queries': 9}


In [2]:
def canonical_url(value):
    parts = urlsplit(value)
    return urlunsplit((parts.scheme, parts.netloc.lower(), parts.path.rstrip('/') or '/', parts.query, ''))

def png_dimensions(path):
    raw = path.read_bytes()[:24]
    if raw[:8] != b'\x89PNG\r\n\x1a\n':
        raise ValueError(f'Not PNG: {path}')
    return struct.unpack('>II', raw[16:24])

rows = []
for query, results in SERP.items():
    stem = query.replace(' ', '_')
    html_path = EVIDENCE / f'{stem}.html'
    png_path = EVIDENCE / f'{stem}.png'
    html = html_path.read_text(errors='ignore')
    width, height = png_dimensions(png_path)
    canonical = [canonical_url(item['url']) for item in results]
    rows.append({
        'query': query,
        'organic_rows': len(results),
        'unique_canonical_urls': len(set(canonical)),
        'missing_snippets': sum(not item.get('snippet') for item in results),
        'has_html': html_path.is_file(),
        'has_png': png_path.is_file(),
        'gl_us': 'gl=us' in html,
        'hl_en': 'hl=en' in html,
        'pws_0': 'pws=0' in html,
        'png_width': width,
        'png_height': height,
    })
profile = pd.DataFrame(rows).sort_values('query').reset_index(drop=True)
profile

,query,organic_rows,unique_canonical_urls,missing_snippets,has_html,has_png,gl_us,hl_en,pws_0,png_width,png_height
0,ai lead capture form builder,9,9,2,True,True,True,True,True,1265,3858
1,cheaper typeform alternative,8,8,3,True,True,True,True,True,1265,4286
2,free typeform alternative,10,10,3,True,True,True,True,True,1265,4547
3,how to create a qr code form,9,8,3,True,True,True,True,True,1265,5162
4,lead capture form builder,10,10,1,True,True,True,True,True,1265,5020
5,lead form ai,7,7,3,True,True,True,True,True,1265,5584
6,qr code event registration form,9,9,4,True,True,True,True,True,1265,4880
7,qr code form builder,8,8,5,True,True,True,True,True,1265,4243
8,typeform alternative free,10,10,3,True,True,True,True,True,1265,4809


## Results

In [3]:
summary = {
    'queries': len(profile),
    'html_files': len(list(EVIDENCE.glob('*.html'))),
    'png_files': len(list(EVIDENCE.glob('*.png'))),
    'organic_rows': int(profile.organic_rows.sum()),
    'queries_with_10_rows': int((profile.organic_rows == 10).sum()),
    'queries_with_7_to_9_rows': int(profile.organic_rows.between(7, 9).sum()),
    'missing_snippets': int(profile.missing_snippets.sum()),
    'all_us_markers': bool(profile[['gl_us', 'hl_en', 'pws_0']].all().all()),
    'all_full_page_images': bool((profile.png_height > 3800).all() and (profile.png_width == 1265).all()),
}
print(json.dumps(summary, indent=2))

{
  "queries": 9,
  "html_files": 9,
  "png_files": 9,
  "organic_rows": 80,
  "queries_with_10_rows": 3,
  "queries_with_7_to_9_rows": 6,
  "missing_snippets": 27,
  "all_us_markers": true,
  "all_full_page_images": true
}


## Takeaways

1. Evidence capture passes completeness and region-marker checks.
2. Rename the evidence claim from `Top 10 for every query` to `visible first-page organic results, up to 10`.
3. Missing snippet text limits copy-level analysis, but titles, URLs, ordering, ads, AI Overview, and screenshots are sufficient for intent classification.
4. Manual screenshot review overrides the reports where QR/check-in, Lead AI/qualification, and Typeform page-type distributions were overstated.